# Content-Based Recommendation System

**Author:** Daniela Balaniuc

**Project:** NewsLens AI

**Notebook:** 04 - Content-Based Recommender

## Objective

Build a content-based news recommendation system using TF-IDF vectorization and cosine similarity on the Microsoft MIND dataset.

This recommender represents each article using TF-IDF vectors derived from the title and abstract. Recommendations are generated by computing cosine similarity between the selected article and all other articles, returning the most similar ones.

Imports

In [1]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import MINDDataLoader

## Load the Dataset

In [2]:
loader = MINDDataLoader(
    PROJECT_ROOT / "data" / "raw" / "MINDsmall"
)

news, behaviors = loader.load()

print(news.shape)
print(behaviors.shape)

(51282, 8)
(156965, 5)


## Prepare Article Text

In [3]:
news["text"] = (
    news["title"].fillna("")
    + " "
    + news["abstract"].fillna("")
)

In [4]:
news[["title", "text"]].head()

,title,text
0,"The Brands Queen Elizabeth, Prince Charles, an...","The Brands Queen Elizabeth, Prince Charles, an..."
1,50 Worst Habits For Belly Fat,50 Worst Habits For Belly Fat These seemingly ...
2,The Cost of Trump's Aid Freeze in the Trenches...,The Cost of Trump's Aid Freeze in the Trenches...
3,I Was An NBA Wife. Here's How It Affected My M...,I Was An NBA Wife. Here's How It Affected My M...
4,"How to Get Rid of Skin Tags, According to a De...","How to Get Rid of Skin Tags, According to a De..."


Inspect Data

In [5]:
news["text"].isna().sum()

np.int64(0)

Display an example

In [6]:
news.loc[0, "text"]

"The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By Shop the notebooks, jackets, and more that the royals can't live without."

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

## TF-IDF Vectorization

In [9]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(news["text"])

Inspect the matrix

In [10]:
print(tfidf_matrix.shape)

(51282, 5000)


Look at the vocabulary

In [11]:
feature_names = vectorizer.get_feature_names_out()

feature_names[:20]

array(['00', '000', '10', '100', '101', '1010', '106', '108', '10th',
       '11', '112', '11th', '12', '120', '13', '14', '140', '14th', '15',
       '150'], dtype=object)

Check sparsity

In [12]:
print(f"Non-zero values: {tfidf_matrix.nnz:,}")
print(f"Total values: {tfidf_matrix.shape[0] * tfidf_matrix.shape[1]:,}")

Non-zero values: 904,170
Total values: 256,410,000


## TF-IDF Representation

Each article is transformed into a numerical vector using Term Frequency–Inverse Document Frequency (TF-IDF).

TF-IDF increases the importance of words that are distinctive to an article while reducing the weight of common words shared across many articles.

These vectors provide the numerical representation used to compare article similarity.

Import the Recommender

In [13]:
from src.recommenders.content_based import ContentBasedRecommender

Train

In [14]:
model = ContentBasedRecommender()

model.fit(news)

Cosine Similarity

In [15]:
news.loc[0, "text"]

"The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By Shop the notebooks, jackets, and more that the royals can't live without."


## Build the Recommender

Pick one article

In [16]:
news.iloc[0]

news_id                                                         N55528
category                                                     lifestyle
subcategory                                            lifestyleroyals
title                The Brands Queen Elizabeth, Prince Charles, an...
abstract             Shop the notebooks, jackets, and more that the...
url                      https://assets.msn.com/labs/mind/AAGH0ET.html
title_entities       [{"Label": "Prince Philip, Duke of Edinburgh",...
abstract_entities                                                   []
text                 The Brands Queen Elizabeth, Prince Charles, an...
Name: 0, dtype: object

Recommend

In [17]:
model.recommend("N55528")

,news_id,title,category,similarity
28360,N9056,This Is What Queen Elizabeth Is Doing About th...,lifestyle,0.605993
31743,N38133,The cutest photos of royal children and their ...,lifestyle,0.527236
29974,N60671,Prince Charles Teared Up When Prince William T...,lifestyle,0.512445
38035,N43522,Prince Charles is Getting Into Fashion,lifestyle,0.489382
35826,N51725,Prince Charles Looks in Awe of Master Archie a...,video,0.468659


In [18]:
recommendations = model.recommend("N55528")

recommendations

,news_id,title,category,similarity
28360,N9056,This Is What Queen Elizabeth Is Doing About th...,lifestyle,0.605993
31743,N38133,The cutest photos of royal children and their ...,lifestyle,0.527236
29974,N60671,Prince Charles Teared Up When Prince William T...,lifestyle,0.512445
38035,N43522,Prince Charles is Getting Into Fashion,lifestyle,0.489382
35826,N51725,Prince Charles Looks in Awe of Master Archie a...,video,0.468659


In [19]:
selected = news.loc[news["news_id"] == "N55528"]

print("Selected Article")
display(selected[["title", "category"]])

Selected Article


,title,category
0,"The Brands Queen Elizabeth, Prince Charles, an...",lifestyle


## Results

The recommender successfully identified articles related to the selected news story.

The recommendations focus on the same entities and topics, including Queen Elizabeth, Prince Charles, Prince William, and the British Royal Family.

Unlike the popularity-based recommender, this model adapts its recommendations according to the content of the selected article, providing a more personalized experience.

## Example Recommendations

Sports

In [20]:
sports_article = news.loc[
    news["category"] == "sports",
    "news_id"
].iloc[0]

sports_article

'N2073'

In [21]:
model.recommend(sports_article)

,news_id,title,category,similarity
4441,N61576,NFL fines Baker Mayfield for stating the obvious,sports,0.486028
28296,N29891,NFL officiating stinks. Here are 10 ways to fi...,sports,0.414395
450,N46662,NFL Cheerleaders,sports,0.352650
26368,N51783,Retired Eagles DE Chris Long calls officiating...,sports,0.351887
13765,N3314,5 NFL breakout players of 2019,sports,0.351358


Finance

In [22]:
finance_article = news.loc[
    news["category"] == "finance",
    "news_id"
].iloc[0]

model.recommend(finance_article)

,news_id,title,category,similarity
3286,N47813,The Live Mascots of College Football,sports,0.407036
42505,N24953,2 RI Cities Named Among Best Places To Live On...,travel,0.374691
5862,N61217,Billionaires who live in the smallest American...,finance,0.371567
2320,N62095,25 Places Where Cars Are Not Allowed,travel,0.344854
39633,N11298,Live look at roads: Traffic cameras around Wes...,autos,0.338296


Health

In [23]:
health_article = news.loc[
    news["category"] == "health",
    "news_id"
].iloc[0]

model.recommend(health_article)

,news_id,title,category,similarity
6172,N47331,Discouraged From Trying to Lose Belly Fat and ...,health,0.565403
154,N60584,Those Grueling Workouts May Not Help You Lose ...,health,0.538480
292,N16032,"If You Have a Slow Metabolism, Here Are 5 Doct...",health,0.536611
21176,N16912,"Rowing Can Help You Burn Belly Fat, but You'll...",health,0.514504
6440,N56301,10 Ways to Burn Belly Fat in 10 Minutes,health,0.503859


## Comparison with the Popularity Baseline

| Popularity-Based                  | Content-Based                |
| --------------------------------- | ---------------------------- |
| Same recommendations for everyone | Personalized by article      |
| Uses click counts                 | Uses article text            |
| Easy to implement                 | Uses NLP                     |
| No personalization                | Personalized recommendations |


## Discussion

The content-based recommender successfully identifies articles with similar themes by comparing textual features extracted from article titles and abstracts.

Compared with the popularity baseline, this model provides personalized recommendations that depend on the selected article rather than overall click counts.

This approach demonstrates how Natural Language Processing techniques such as TF-IDF and cosine similarity can be applied to build practical recommendation systems.

## Next Steps

- Build a collaborative filtering model using user interaction data.
- Compare collaborative filtering with content-based recommendations.
- Evaluate recommendation quality using ranking metrics such as Precision@K and Recall@K.
- Explore hybrid recommendation approaches that combine content and user behavior.